In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import random
from tensorflow.keras.layers import Input, Dense, LayerNormalization, MultiHeadAttention, Dropout, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import glob

# ✅ 시드 고정
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# 시퀀스 길이 설정 (예: 과거 30일 데이터를 보고 다음 날 예측)
sequence_length = 60

# 기술적 지표 목록
technical_indicators = [
    "sma_50", "ema_20", "wma_20", "macd", "macd_signal", "macd_hist", "tema_20",
    "rsi_14", "roc", "cci_14", "willr_14", "atr_14", "upper_bb", "middle_bb", "lower_bb",
    "obv", "ad", "chaikin_ad"
]

# 데이터 불러오기
csv_files = glob.glob("*_Train.csv")
df = pd.read_csv(csv_files[0])  # BTC 데이터 사용

# 독립 변수(X)와 종속 변수(y) 분리
X = df[technical_indicators].values
y = df["label"].values

# 🔹 라벨 값 변경 (-1 → 2, 0 → 0, 1 → 1)
y = y + 1

# 데이터 정규화
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 시계열 데이터로 변환
def create_sequences(X, y, seq_length):
    Xs, ys = [], []
    for i in range(len(X) - seq_length):
        Xs.append(X[i : i + seq_length])
        ys.append(y[i + seq_length])
    return np.array(Xs), np.array(ys)

X_seq, y_seq = create_sequences(X_scaled, y, sequence_length)

# 훈련/테스트 데이터 분할 (80:20 비율, ✅ 시드 고정)
X_train, X_test, y_train, y_test = train_test_split(
    X_seq, y_seq, test_size=0.2, random_state=SEED, stratify=y_seq
)

# ✅ 시드 고정 (TensorFlow 실행)
tf.random.set_seed(SEED)

# Transformer Encoder Block
def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0.1):
    x = LayerNormalization(epsilon=1e-6)(inputs)
    x = MultiHeadAttention(key_dim=head_size, num_heads=num_heads)(x, x)
    x = Dropout(dropout)(x)
    res = x + inputs

    x = LayerNormalization(epsilon=1e-6)(res)
    x = Dense(ff_dim, activation="relu")(x)
    x = Dropout(dropout)(x)
    x = Dense(inputs.shape[-1], activation="linear")(x)
    return x + res

# 🔹 Transformer 모델 생성 (차원 맞추기 수정)
input_layer = Input(shape=(sequence_length, len(technical_indicators)))
x = transformer_encoder(input_layer, head_size=64, num_heads=4, ff_dim=128)
x = transformer_encoder(x, head_size=64, num_heads=4, ff_dim=128)

# 🔹 시퀀스 전체를 하나의 벡터로 변환 (GlobalAveragePooling1D)
x = GlobalAveragePooling1D()(x)
x = Dense(32, activation="relu")(x)
x = Dropout(0.2)(x)

# 🔹 최종 예측 (3개의 클래스)
output_layer = Dense(3, activation="softmax")(x)

transformer_model = Model(inputs=input_layer, outputs=output_layer)

# 모델 컴파일 및 학습
transformer_model.compile(optimizer=Adam(learning_rate=0.001), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history_transformer = transformer_model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test))

# 🔹 학습 데이터 저장
train_loss_transformer = history_transformer.history['loss'][-1]
train_accuracy_transformer = history_transformer.history['accuracy'][-1]

# Transformer 모델을 .keras 형식으로 저장
transformer_model.save("Transformer_model.keras", save_format="keras")
print("Transformer 모델 저장 완료! (Keras 포맷)")


Epoch 1/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 8s 47ms/step - accuracy: 0.3160 - loss: 1.1258 - val_accuracy: 0.3551 - val_loss: 1.1023
Epoch 2/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.3564 - loss: 1.1050 - val_accuracy: 0.3514 - val_loss: 1.0975
Epoch 3/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.3225 - loss: 1.1025 - val_accuracy: 0.3514 - val_loss: 1.0968
Epoch 4/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 0.3340 - loss: 1.1089 - val_accuracy: 0.3551 - val_loss: 1.0996
Epoch 5/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.3263 - loss: 1.0999 - val_accuracy: 0.3551 - val_loss: 1.0962
Epoch 6/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.3314 - loss: 1.1008 - val_accuracy: 0.3514 - val_loss: 1.0970
Epoch 7/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.3507 - loss: 1.0995 - val_accuracy: 0.3623 - val_loss: 1.0939
Epoch 8/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.3288 - loss: 1.1000 - val_accuracy: 0.3587 - v

Transformer 모델 저장 완료! (Keras 포맷)
